# Gradient Boosting

Learn this model in order: **setup → baseline → initial model → tuning → interpretation**.

We keep all 395 students, including 38 recorded zero grades. The same 316 training students and five folds are used in every notebook. The 79 test students belong to final evaluation in Notebook 04.

There are two clearly separated scenarios: without G1/G2 (main) and with both grades (later stage). Input timing, particularly absences, still needs verification. MAE remains the selection metric. The test results have already been inspected: rerunning the unchanged experiment reproduces it, rather than creating a new unseen test.

Results and explanations stay in this notebook. Only fitted pipelines are saved to `models/` for the comparison and evaluation notebooks.

## 1 Setup

Shared helpers in `src/model_utils.py` handle the split and repeated metric calculations. Model settings, fitting and tuning remain visible here. Preprocessing is always fitted inside the training fold.

MAE/RMSE measure grade-point error; R² describes squared-error performance. Accuracy, precision, recall, F1 and specificity evaluate the derived **predicted G3 < 10** flag against **actual G3 < 10**. No rounding or threshold tuning is used.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


In [2]:
MODEL_NAME = 'Gradient Boosting'
MODEL_SLUG = 'gradient_boosting'
SCALE_NUMERIC = False

from sklearn.ensemble import GradientBoostingRegressor
INITIAL_ESTIMATOR = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=2, min_samples_leaf=5, random_state=42)
MODEL_PARAM_GRID = {'model__n_estimators': [100, 200], 'model__learning_rate': [0.03, 0.05, 0.1], 'model__max_depth': [1, 2], 'model__min_samples_leaf': [5, 10]}

## 2 Constant baselines

Mean and median baselines learn their constant from each fold's training labels. They ignore the inputs, so one set of scores applies to both scenarios. Precision is undefined when nobody is flagged; missing values are left visible rather than replaced with perfect scores.

In [3]:
baseline_folds = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(DummyRegressor(strategy=strategy), X_no_grades, y_train,
        cv=CV, scoring=SCORING, error_score="raise")
    baseline_folds[strategy] = fold_results(scores)

display(pd.DataFrame({name: folds[METRICS].mean()
    for name, folds in baseline_folds.items()}).T.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4029,4.5635,-0.0350,0.6744,NaN,0.0,0.0,1.0
median,3.3403,4.5931,-0.0519,0.6744,NaN,0.0,0.0,1.0


## 3 Without G1 and G2

This is the main prediction scenario. Excluding grades does not by itself prove that all other inputs are available early.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [4]:
initial_no_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=False)
initial_scores_no_grades = cross_validate(initial_no_grades, X_no_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_no_grades = fold_results(initial_scores_no_grades)
display(cv_summary(initial_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.0461,3.9459,0.2102,0.6900,0.5317,0.4819,0.4910,0.7957
std,0.4100,0.3768,0.1190,0.0722,0.1562,0.1892,0.1372,0.0806
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [5]:
grid_no_grades = dict(MODEL_PARAM_GRID)
grid_no_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=False)
    for use_log in [False, True]
]
search_no_grades = GridSearchCV(initial_no_grades, grid_no_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_no_grades.fit(X_no_grades, y_train)
best_no_grades = search_no_grades.best_estimator_
tuned_folds_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_)

print("Model settings:", best_no_grades.named_steps["model"])
print("Log absences:", best_no_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.0461,3.9459,0.2102,0.6900,0.5317,0.4819,0.4910,0.7957
std,0.4100,0.3768,0.1190,0.0722,0.1562,0.1892,0.1372,0.0806
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: GradientBoostingRegressor(learning_rate=0.05, max_depth=2, min_samples_leaf=5,
                          random_state=42)
Log absences: False


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [6]:
comparison_no_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_no_grades[METRICS].mean(),
    "Tuned model": tuned_folds_no_grades[METRICS].mean(),
}).T
display(comparison_no_grades.round(4))

training_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_no_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_no_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_no_grades),
    "Tuned model": pooled_risk(tuned_folds_no_grades)}).T.round(4))
display(tuned_folds_no_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.000,1.0000
Initial model,3.0461,3.9459,0.2102,0.6900,0.5317,0.4819,0.491,0.7957
Tuned model,3.0461,3.9459,0.2102,0.6900,0.5317,0.4819,0.491,0.7957


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.6899,0.5269,0.4757,0.5,0.7934,49.0,44.0,54.0,169.0
Tuned model,0.6899,0.5269,0.4757,0.5,0.7934,49.0,44.0,54.0,169.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,2.5959,3.5839,0.1054,0.6406,0.6667,0.3571,0.4651,0.8611,31.0,5.0,18.0,10.0
1,2,3.6117,4.4516,0.3419,0.7302,0.5714,0.7619,0.6531,0.7143,30.0,12.0,5.0,16.0
2,3,2.7277,3.5572,0.0991,0.5873,0.2778,0.2778,0.2778,0.7111,32.0,13.0,13.0,5.0
3,4,3.2652,4.0570,0.1734,0.7460,0.5000,0.5625,0.5294,0.8085,38.0,9.0,7.0,9.0
4,5,3.0299,4.0797,0.3310,0.7460,0.6429,0.4500,0.5294,0.8837,38.0,5.0,11.0,9.0


Tuned training MAE: 2.45
Tuned validation MAE: 3.0461


## 4 With G1 and G2

This is a later-stage comparison, usable only once both prior grades are known. It answers a different prediction-timing question.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [7]:
initial_with_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=True)
initial_scores_with_grades = cross_validate(initial_with_grades, X_with_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_with_grades = fold_results(initial_scores_with_grades)
display(cv_summary(initial_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,0.9849,1.5800,0.8765,0.8703,0.7190,0.9889,0.8304,0.8101
std,0.2096,0.4625,0.0377,0.0504,0.0943,0.0248,0.0695,0.0747
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [8]:
grid_with_grades = dict(MODEL_PARAM_GRID)
grid_with_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=True)
    for use_log in [False, True]
]
search_with_grades = GridSearchCV(initial_with_grades, grid_with_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_with_grades.fit(X_with_grades, y_train)
best_with_grades = search_with_grades.best_estimator_
tuned_folds_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_)

print("Model settings:", best_with_grades.named_steps["model"])
print("Log absences:", best_with_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,0.9849,1.5800,0.8765,0.8703,0.7190,0.9889,0.8304,0.8101
std,0.2096,0.4625,0.0377,0.0504,0.0943,0.0248,0.0695,0.0747
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: GradientBoostingRegressor(learning_rate=0.05, max_depth=2, min_samples_leaf=5,
                          random_state=42)
Log absences: False


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [9]:
comparison_with_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_with_grades[METRICS].mean(),
    "Tuned model": tuned_folds_with_grades[METRICS].mean(),
}).T
display(comparison_with_grades.round(4))

training_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_with_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_with_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_with_grades),
    "Tuned model": pooled_risk(tuned_folds_with_grades)}).T.round(4))
display(tuned_folds_with_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,0.9849,1.5800,0.8765,0.8703,0.719,0.9889,0.8304,0.8101
Tuned model,0.9849,1.5800,0.8765,0.8703,0.719,0.9889,0.8304,0.8101


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.8703,0.7183,0.9903,0.8327,0.8122,102.0,40.0,1.0,173.0
Tuned model,0.8703,0.7183,0.9903,0.8327,0.8122,102.0,40.0,1.0,173.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,0.9710,1.4025,0.8630,0.8438,0.7368,1.0000,0.8485,0.7222,26.0,10.0,0.0,28.0
1,2,1.3196,2.2898,0.8259,0.9048,0.7778,1.0000,0.8750,0.8571,36.0,6.0,0.0,21.0
2,3,0.8062,1.2116,0.8955,0.8095,0.6071,0.9444,0.7391,0.7556,34.0,11.0,1.0,17.0
3,4,0.8104,1.2054,0.9270,0.8571,0.6400,1.0000,0.7805,0.8085,38.0,9.0,0.0,16.0
4,5,1.0172,1.7907,0.8711,0.9365,0.8333,1.0000,0.9091,0.9070,39.0,4.0,0.0,20.0


Tuned training MAE: 0.7774
Tuned validation MAE: 0.9849


## 5 Compare scenarios and save the fitted models

The score difference reflects the information available at different prediction times. Do not present the G1/G2 scenario as an early prediction. Save only the two complete fitted pipelines; tables and findings remain inside the notebook.

In [10]:
display(pd.DataFrame({
    "Without G1/G2": tuned_folds_no_grades[METRICS].mean(),
    "With G1/G2": tuned_folds_with_grades[METRICS].mean(),
}).T.round(4))

model_dir = PROJECT_ROOT / "models/experiments"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_no_grades, model_dir / "gradient_boosting__no_prior_grades.joblib")
joblib.dump(best_with_grades, model_dir / "gradient_boosting__with_prior_grades.joblib")
print("Saved fitted pipelines. No experiment tables or reports are exported.")

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Without G1/G2,3.0461,3.9459,0.2102,0.6900,0.5317,0.4819,0.4910,0.7957
With G1/G2,0.9849,1.5800,0.8765,0.8703,0.7190,0.9889,0.8304,0.8101


Saved fitted pipelines. No experiment tables or reports are exported.
